In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import cv2

In [ ]:
dataset, info=tfds.load('oxford_iiit_pet:4.*.*',with_info=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.LU3CJJ_4.0.0/oxford_iiit_pet-train.tfrecord*...…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/oxford_iiit_pet/incomplete.LU3CJJ_4.0.0/oxford_iiit_pet-test.tfrecord*...:…

Dataset oxford_iiit_pet downloaded and prepared to /root/tensorflow_datasets/oxford_iiit_pet/4.0.0. Subsequent calls will reuse this data.


In [ ]:
train_data=dataset['train']
test_data=dataset['test']

In [ ]:
IMG_SIZE=128
BATCH_SIZE=16

In [ ]:
def preprocess(data):
  image=tf.image.resize(data['image'],(IMG_SIZE,IMG_SIZE))
  mask=tf.image.resize(data['segmentation_mask'],(IMG_SIZE,IMG_SIZE))
  image=tf.cast(image,tf.float32)
  image=tf.keras.applications.mobilenet_v2.preprocess_input(image)
  mask=tf.cast(mask,tf.int32)-1
  return image,mask

In [ ]:
test_dataset=(test_data.map(preprocess,num_parallel_calls=tf.data.AUTOTUNE).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))
train_dataset=(train_data.map(preprocess,num_parallel_calls=tf.data.AUTOTUNE).cache().shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

def build_pretrained_unet(output_channels=3):
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(128, 128, 3),
        include_top=False,
        weights="imagenet"
    )

    layer_names = [
        "block_1_expand_relu",   # 64x64
        "block_3_expand_relu",   # 32x32
        "block_6_expand_relu",   # 16x16
        "block_13_expand_relu",  # 8x8
        "block_16_project"       # 4x4
    ]

    # Encoder outputs
    skip_outputs = [base_model.get_layer(name).output for name in layer_names]
    encoder = tf.keras.Model(inputs=base_model.input, outputs=skip_outputs)
    encoder.trainable = False

    inputs = layers.Input(shape=(128, 128, 3))
    skips = encoder(inputs)

    x = skips[-1]
    skip_connections = list(reversed(skips[:-1]))

    up_filters = [512, 256, 128, 64]

    # Decoder
    for filters, skip in zip(up_filters, skip_connections):
        x = layers.Conv2DTranspose(filters, 3, strides=2, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.Concatenate()([x, skip])

        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)

    # Final upsampling to restore 128x128
    x = layers.Conv2DTranspose(32, 3, strides=2, padding="same")(x)
    x = layers.ReLU()(x)

    outputs = layers.Conv2D(output_channels, 1, activation="softmax")(x)

    return tf.keras.Model(inputs, outputs)

In [ ]:
pretrained_model=build_pretrained_unet(3)
pretrained_model.summary()

Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_13      │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_8        │ [(None, 64, 64,   │  1,841,984 │ input_layer_13[0… │
│ (Functional)        │ 96), (None, 32,   │            │                   │
│                     │ 32, 144), (None,  │            │                   │
│                     │ 16, 16, 192),     │            │                   │
│                     │ (None, 8, 8,      │            │                   │
│                     │ 576), (None, 4,   │            │                   │
│                     │ 4, 320)]          │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_15 │ (None, 8, 8, 512) │  1,475,072 │ functional_8[0][… │
│ (Conv2DTranspose)   │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 8, 8, 512) │      2,048 │ conv2d_transpose… │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_15 (ReLU)     │ (None, 8, 8, 512) │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_15      │ (None, 8, 8,      │          0 │ re_lu_15[0][0],   │
│ (Concatenate)       │ 1088)             │            │ functional_8[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_34 (Conv2D)  │ (None, 8, 8, 512) │  5,014,016 │ concatenate_15[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_35 (Conv2D)  │ (None, 8, 8, 512) │  2,359,808 │ conv2d_34[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_16 │ (None, 16, 16,    │  1,179,904 │ conv2d_35[0][0]   │
│ (Conv2DTranspose)   │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │      1,024 │ conv2d_transpose… │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ re_lu_16 (ReLU)     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_16      │ (None, 16, 16,    │          0 │ re_lu_16[0][0],   │
│ (Concatenate)       │ 448)              │            │ functional_8[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_36 (Conv2D)  │ (None, 16, 16,    │  1,032,448 │ concatenate_16[0… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_37 (Conv2D)  │ (None, 16, 16,    │    590,080 │ conv2d_36[0][0]   │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_17 │ (None, 32, 32,    │    295,040 │ conv2d_37[0][0]   │
│ (Conv2DTranspose)   │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        512 │ conv2d_transpose… │
│ (BatchNormalizatio… │ 128)              │            │                 

 Total params: 14,474,755 (55.22 MB)

 Trainable params: 12,630,851 (48.18 MB)

 Non-trainable params: 1,843,904 (7.03 MB)

In [ ]:
pretrained_model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                     loss='sparse_categorical_crossentropy',
                     metrics=['accuracy'])

In [ ]:
history=pretrained_model.fit(train_dataset,validation_data=test_dataset,epochs=5)

Epoch 1/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 91s 224ms/step - accuracy: 0.8007 - loss: 0.4854 - val_accuracy: 0.8588 - val_loss: 0.4226
Epoch 2/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 23s 101ms/step - accuracy: 0.8968 - loss: 0.2741 - val_accuracy: 0.9061 - val_loss: 0.2545
Epoch 3/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 23s 99ms/step - accuracy: 0.9049 - loss: 0.2506 - val_accuracy: 0.9013 - val_loss: 0.2798
Epoch 4/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 22s 95ms/step - accuracy: 0.9124 - loss: 0.2314 - val_accuracy: 0.9081 - val_loss: 0.2527
Epoch 5/5
230/230 ━━━━━━━━━━━━━━━━━━━━ 22s 94ms/step - accuracy: 0.9168 - loss: 0.2164 - val_accuracy: 0.9103 - val_loss: 0.2532
